In [73]:
import pandas as pd 
import numpy as np 

In [74]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest,chi2

In [75]:
df=pd.read_csv('tested.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [76]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

In [77]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,34.5,0,0,7.8292,Q
1,1,3,female,47.0,1,0,7.0000,S
2,0,2,male,62.0,0,0,9.6875,Q
3,0,3,male,27.0,0,0,8.6625,S
4,1,3,female,22.0,1,1,12.2875,S


In [78]:
x_train,x_test,y_train,y_test=train_test_split(df.drop(columns=['Survived']),df['Survived'],test_size=0.3,random_state=0)

In [79]:
x_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
96,1,female,76.0,1,0,78.8500,S
381,3,male,26.0,0,0,7.8792,Q
89,2,male,2.0,1,1,23.0000,S
233,3,male,NaN,0,0,7.8792,Q
191,1,male,NaN,0,0,26.0000,S


In [80]:
y_train.head()

96     1
381    0
89     0
233    0
191    0
Name: Survived, dtype: int64

In [81]:
df.isnull().sum()

Survived     0
Pclass       0
Sex          0
Age         86
SibSp        0
Parch        0
Fare         1
Embarked     0
dtype: int64

In [94]:
# imputstion transformer
trf1=ColumnTransformer([
    ('impute_age',SimpleImputer(),[2]),
 ('impute_fare',SimpleImputer(),[5])
],remainder='passthrough')

In [106]:
#ohe on sex,embarked
trf2=ColumnTransformer([
    ('ohe_sex_embarked',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),[1,6])
])

In [95]:
#scaling 
trf3=ColumnTransformer([
    ('scale',MinMaxScaler(),slice(0,10))
])

In [118]:
#feature selection
trf4=SelectKBest(score_func=chi2,k=5)

In [98]:
trf5=DecisionTreeClassifier()

In [119]:
pipe=Pipeline([
    ('trf1',trf1),
    ('trf2',trf2),
    ('trf3',trf3),
    ('trf4',trf4),
    ('trf5',trf5)
])

pipeline vs make_pipeline
make_pipeline is easier than pipeline because we dont need to write two para 
the same thing is apply in columntrans.vs make_columnt. here we pass two para
we dont pass 'imputeage'

In [120]:
pipe=make_pipeline(trf1,trf2,trf3,trf4,trf5)

In [121]:
pipe.fit(x_train,y_train)
# here we import model(decisiontree) so we can only fit the data 
# if we dont import model then we haveto apply fit_transform

Pipeline(steps=[('columntransformer-1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_fare',
                                                  SimpleImputer(), [5])])),
                ('columntransformer-2',
                 ColumnTransformer(transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('columntransformer-3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('selectkbest',
                 SelectKBest(k=5,
                             score_func=<function chi2 at 0x000002180591C220>)),
                ('decisiontreeclassifier', DecisionTreeClassifier())])

In [102]:
pipe.named_steps

{'columntransformer-1': ColumnTransformer(remainder='passthrough',
                   transformers=[('impute_age', SimpleImputer(), [2]),
                                 ('impute_fare', SimpleImputer(), [5])]),
 'columntransformer-2': ColumnTransformer(transformers=[('ohe_sex_embarked',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse_output=False),
                                  [3, 6])]),
 'columntransformer-3': ColumnTransformer(transformers=[('scale', MinMaxScaler(), slice(0, 10, None))]),
 'selectkbest': SelectKBest(k=8, score_func=<function chi2 at 0x000002180591C220>),
 'decisiontreeclassifier': DecisionTreeClassifier()}

In [103]:
# display pipeline
from sklearn import set_config
set_config(display='diagram')

In [108]:
# predict
y_pred=pipe.predict(x_test)

In [110]:
y_pred

array([0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1,
       0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1,
       0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1,
       1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1,
       1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0,
       0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0], dtype=int64)

In [123]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

1.0

cross validation using pipeline

In [124]:
from sklearn.model_selection import cross_val_score
cross_val_score(pipe,x_train,y_train,cv=5,scoring='accuracy').mean()

0.667855055523086

exporting pipeline

In [127]:
import pickle
pickle.dump(pipe,open('pipe.pkl','wb'))